# S5.M7-1 主动方明文直方图（M3 后半）与 SGB 开销

**目的**：方法筛选第四阶段第 1 步（[DR-M5-015](../../../registry/decision_records/DR-M5-015.yaml) · D-1 ~ D-5）。

隐语 1.13 `sgb_v` 默认模式会把同一份梯度密文也发给主动方自己。于是主动方自己的特征也走密文分桶求和，然后再解密。但主动方本来就持有明文梯度和私钥，这一段加密什么也不保护。

本步在实验代码里把主动方那一份换成明文，这正是 SecureBoost（Cheng 等，IEEE Intelligent Systems 2021）的原始设计。检验三件事：
- 与原版是否逐位一致：6 个任务完整拟合，与 S5.M2 存档的测试打分逐行比较；
- 两方合计计算时间能省多少：ABBA 顺序，前 10 棵树；
- 预筛列数的 SGB 单树开销：Orange 大版流失，每方 k 列。

**实现**：只替换 `GradientEncryptor.cache_to_workers`，不改隐语源码。
- 被动方收到的梯度密文与原实现相同（同一次加密）。
- 主动方收到自己明文梯度的 HEU 明文对象，做法与隐语 `label_holder_feature_only` 分支相同。
- 其余步骤不动：分桶、打乱、兄弟节点相减、求分裂。

**口径**：在会话容器上做 debug_mode 单进程两方模拟（OU），两方的计算串行相加。测得的是两方合计计算时间，不是真实部署的墙钟时长。

**数据**：公开数据，属开发证据。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m7_sgb_cost.yaml"
m7 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m7["base_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("sgb_cost_screen.py", "method_screen_sgb.py", "method_screen.py", "method_screen_large.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m7["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m7["decision_record"], "| step:", m7["step_id"], "|", m7["purpose"])
print("seed:", m7["seed"], "（每个 SGB 拟合一次）| git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
SGB, LARGE, COST = (pathlib.Path(os.environ[m7[k]]) for k in ("sgb_dir_env", "large_dir_env", "cost_dir_env"))
print("启动指纹（每段运行开始时记录；路径只留文件名）：")
launch = {}
for line in (COST / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    if "sha256:" in line:
        path, digest = line.split(" sha256:")
        launch.setdefault(pathlib.Path(path).name, set()).add(digest)
        print("  sha256:" + digest[:HASH_PREFIX_LEN], pathlib.Path(path).name)
    else:
        print(" ", line)
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == {hashlib.sha256(p.read_bytes()).hexdigest()}, p.name
print("各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。")

modules/m5_modeling/configs/s5_m7_sgb_cost.yaml sha256:94dcd565530f
registry/decision_records/DR-M5-015.yaml sha256:078b1b1ee1c4
modules/m5_modeling/components/sgb_cost_screen.py sha256:cf6088aa95be
modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
modules/m5_modeling/components/method_screen_large.py sha256:5121a5b9368c
决策记录: registry/decision_records/DR-M5-015.yaml | step: S5.M7 | 主动方明文直方图与原版是否逐位一致、省多少计算；Orange 大版上每方 k 列的 SGB 单树开销
seed: 11 （每个 SGB 拟合一次）| git: 1cf98ce（工作区含本步未提交改动）
启动指纹（每段运行开始时记录；路径只留文件名）：
  == 2026-11-04 10:51:39 · part timing ==
  sha256:cf6088aa95be sgb_cost_screen.py
  sha256:94dcd565530f s5_m7_sgb_cost.yaml
  sha256:02027476c924 DR-M5-015.yaml
  sha256:2e54c618305f method_screen_sgb.py
  sha256:05e19e025d0b method_screen.py
  sha256:5121a5b9368c method_screen_large.py
  secretflow 1.13.0b0
  == 2026-11-04 12:10:10 · part kcost ==
  sha256:cf6088aa95be sgb_cost_screen.py
  sha

## 1. 任务与规模

6 个任务列出以下信息：
- 两方列数，以及主动方（内地）列占比；
- 完整拟合棵数（S5.M1 在验证集上选定的配置）；
- 计时棵数；
- 测试行数。

In [2]:
from modules.m5_modeling.components import sgb_cost_screen as SC

EXACT = {n: json.loads((COST / ("exact__%s.json" % n)).read_text(encoding="utf-8")) for n in m7["exact_tasks"]}
TIMING = {n: json.loads((COST / ("timing__%s.json" % n)).read_text(encoding="utf-8")) for n in m7["timing_tasks"]}
KCOST = json.loads((COST / ("kcost__%s.json" % m7["large_task"])).read_text(encoding="utf-8"))
size = pd.DataFrame([{"任务": n, "内地列": t["cols_guest"], "香港列": t["cols_host"], "内地列占比": t["guest_share"],
                      "完整棵数": EXACT[n]["trees"], "计时棵数": t["trees"], "测试行": EXACT[n]["rows"]}
                     for n, t in TIMING.items()])
print(size.round(m7["round_sec"] + 1).to_string(index=False))

              任务  内地列  香港列  内地列占比  完整棵数  计时棵数    测试行
        uci_bank   12    7  0.632    34    10   8238
          fedads   22   23  0.489    14    10 200000
        coil2000   42   43  0.494    59    10   4000
    santander_g3   19   25  0.432    33    10  20351
orange_appetency   80   67  0.544    72    10  10000
  mock_overlap80   20   25  0.444    83    10  17391


## 2. 复现

下一格用仓库里的组件，在 UCI Bank 上重跑明文直方图版本的完整拟合。要求两点：与 S5.M2 存档打分逐位一致；与批量运行的判定相同。

In [3]:
P = SC.MS.Params(cfg)
NAME = m7["exact_tasks"][0]
res = json.loads((RUN / (NAME + ".json")).read_text(encoding="utf-8"))
sc = SC.MS.Screen(NAME, cfg, PRIV, COST / "unused")
again = SC.exact_check(P, sc, res, m7["method"], np.load(SGB / ("%s__%s.npy" % (NAME, m7["method"]))))
print("用仓库组件重跑 %s 明文直方图版本（%d 棵）：与 S5.M2 存档相比 %s，不同 %d / %d 行，测试 AUROC %.6f / %.6f" % (
    NAME, again["trees"], again["verdict"], again["rows_different"], again["rows"], again["auroc_a"], again["auroc_b"]))
assert again["verdict"] == SC.EXACT and again["verdict"] == EXACT[NAME]["verdict"]

[2026-11-04 15:02:51.825] [info] [bigint_spi.cc:79] The default library used for BigInt operations is openssl


[2026-11-04 15:02:59.860] [info] [thread_pool.cc:30] Create a fixed thread pool with size 1


用仓库组件重跑 uci_bank 明文直方图版本（34 棵）：与 S5.M2 存档相比 逐位一致，不同 0 / 8238 行，测试 AUROC 0.805389 / 0.805389


## 3. 结果

- **D-3 逐位比较**：明文版本完整拟合一次，测试集逐行打分（float32）与 S5.M2 存档比较。
- **D-4 计时**：
  - 每个任务先做 2 棵树的预热，再按原版、明文、明文、原版的顺序各拟合一次（前 10 棵树），每次重新初始化；
  - 四次拟合的打分须两两完全相同；
  - 另外分开记录两方各自的分桶求和秒数，以及取回（解密）秒数。
- **D-5 单树开销**：
  - Orange 大版流失，用 S5.M5 选定的优化版联邦配置（深度 3）；
  - 每方 k 列，5 棵树，原版与明文各拟合一次，k 之间交替先后；
  - 外推到全部清洗后列的数字是线性外推值，没有实测。
- 汇总 CSV 写入 `results/sgb_cost_v1/`。

In [4]:
OUT = REPO_ROOT / m7["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R = m7["round_sec"]
exact = pd.DataFrame([{"task": n, "trees": e["trees"], "rows": e["rows"], "rows_different": e["rows_different"],
                       "max_abs_diff": e["max_abs_diff"], "test_auroc_s5m2": e["auroc_a"], "test_auroc_plain": e["auroc_b"],
                       "verdict": e["verdict"], "fit_seconds_plain_full": e["fit_seconds_plain"]} for n, e in EXACT.items()])
timing = pd.DataFrame([{"task": n, "trees": t["trees"], "cols_guest": t["cols_guest"], "cols_host": t["cols_host"],
                        "guest_share": t["guest_share"], "sec_per_tree_base": t["sec_per_tree_base"],
                        "sec_per_tree_plain": t["sec_per_tree_plain"], "saving": t["saving"],
                        "saving_low": t["saving_range"][0], "saving_high": t["saving_range"][1],
                        "preds_identical": t["preds_identical"]} for n, t in TIMING.items()])
runs = pd.DataFrame([{"task": n, "order": i + 1, "kind": r["kind"], "trees": r["trees"], "fit_seconds": r["fit_seconds"],
                      **{k: v for k, v in r["parts"].items()}} for n, t in TIMING.items() for i, r in enumerate(t["runs"])])
kcost = pd.DataFrame([{"task": KCOST["task"], "k_per_party": r["k"], "cols_guest": r["cols_guest"], "cols_host": r["cols_host"],
                       "trees": r["trees"], "sec_per_tree_base": r["sec_per_tree_base"],
                       "sec_per_tree_plain": r["sec_per_tree_plain"], "saving": r["saving"],
                       "preds_identical": r["preds_identical"]} for r in KCOST["rows"]])
large_res = json.loads((LARGE / (KCOST["task"] + ".json")).read_text(encoding="utf-8"))
full_cols = large_res["meta"]["prescreen"]["clean_guest"] + large_res["meta"]["prescreen"]["clean_host"]
cols = (kcost["cols_guest"] + kcost["cols_host"]).tolist()
ext = pd.DataFrame([{"kind": kind, **SC.linear_extrapolate(cols, kcost["sec_per_tree_" + kind].tolist(), full_cols)}
                    for kind in ("base", "plain")])
exact.to_csv(OUT / "exact.csv", index=False, float_format="%.6g")
timing.to_csv(OUT / "timing.csv", index=False, float_format="%.4f")
runs.to_csv(OUT / "timing_runs.csv", index=False, float_format="%.2f")
kcost.to_csv(OUT / "k_cost.csv", index=False, float_format="%.4f")
ext.to_csv(OUT / "k_cost_extrapolation.csv", index=False, float_format="%.4f")
print("D-3 明文直方图与 S5.M2 存档的逐行比较（完整拟合）：")
print(exact.to_string(index=False))
print("判定：", "逐位一致" if (exact["verdict"] == SC.EXACT).all() else sorted(set(exact["verdict"])))
print("D-4 计时（前 n 棵树，ABBA；两方合计计算时间，单进程模拟）：")
print(timing.round(R + 1).to_string(index=False))
print("各次拟合与两方分桶求和、取回秒数：")
print(runs.round(R).to_string(index=False))
print("D-5 Orange 大版流失：每方 k 列的单树秒数，与外推到全部清洗后列（%d 列，外推值）：" % full_cols)
print(kcost.round(R + 1).to_string(index=False))
print(ext.round(R + 2).to_string(index=False))

D-3 明文直方图与 S5.M2 存档的逐行比较（完整拟合）：
            task  trees   rows  rows_different  max_abs_diff  test_auroc_s5m2  test_auroc_plain verdict  fit_seconds_plain_full
        uci_bank     34   8238               0           0.0         0.805389          0.805389    逐位一致                    98.2
          fedads     14 200000               0           0.0         0.641351          0.641351    逐位一致                   334.0
        coil2000     59   4000               0           0.0         0.733060          0.733060    逐位一致                   290.1
    santander_g3     33  20351               0           0.0         0.875382          0.875382    逐位一致                   826.4
orange_appetency     72  10000               0           0.0         0.847585          0.847585    逐位一致                   927.5
  mock_overlap80     83  17391               0           0.0         0.910601          0.910601    逐位一致                  1884.3
判定： 逐位一致
D-4 计时（前 n 棵树，ABBA；两方合计计算时间，单进程模拟）：
            task  trees  co

## 4. 结论

1. **逐位一致（D-3）**：6 个任务用明文直方图版本完整拟合（14 ~ 83 棵树），测试集逐行打分与 S5.M2 存档全部相同（不同 0 行，最大差 0），测试 AUROC 相同。判定「逐位一致」，按 D-6 做 S5.M8。
2. **两方合计计算时间（D-4）**：每棵树省 29% ~ 43%，每个任务四次拟合的打分两两相同。
   - 省下的全在主动方：主动方自己特征上的分桶求和降到原来约 1 / 12，主动方那部分的解密（每 10 棵树 8 ~ 55 秒）降到约 0。
   - 被动方的分桶求和与解密不变，被动方收到的消息也不变。
   - 真实部署中两方并行。主动方的解密只在主动方串行进行，这部分一定省下；分桶求和省下的部分，只在主动方是关键路径时才缩短墙钟时间。这是对协议结构的分析，没有实测。
3. **预筛列数的 SGB 单树开销（D-5）**：Orange 大版流失、深度 3：
   - 每方 100 / 300 / 1000 列时，每棵树原版 25.0 / 73.1 / 241.7 秒，明文 14.6 / 41.7 / 126.8 秒，省 42% ~ 48%；
   - 单树秒数与两方总列数近似成正比，外推到全部清洗后列（13,295 列）原版约 1,730 秒 / 棵（外推，不实测）；
   - 预筛到每方 300 列约为它的 1 / 24，再加明文直方图约 1 / 41。

**对报告的意义**：候选文档 M3 的后半（主动方明文直方图）已完成。模型输出不变，两方合计计算时间省 29% ~ 48%。与 M3 前半（两方预筛，S5.M5）叠加，Orange 大版的单树开销降到全部列原版的约 1 / 41（外推口径）。这是开销上的改进，不改变任何 AUROC 结论。

**边界**：
- 明文直方图只在实验代码里替换一个方法，不改隐语源码；用于正式联调需在隐语上游或项目封装层实现并经安全评审。
- 耗时是单进程两方模拟（OU）上的测量，不代表本机双虚拟机或汇丰环境。
- 公开数据上的开发证据，不换算成汇丰的时长或收益。
- 技术侧自评，未经法务复核（W-001）。